# 05 — Intact Graph Analysis and Binary Network Lesions

Compute modularity and participation coefficient on intact weighted-average FC, on each network subgraph, and after binary virtual lesion of each network.

In [1]:
import json
import multiprocessing as mp
import os
import time
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
CONFIG_NAME = os.environ.get('GRAPH_THEORY_CONFIG', 'config.yaml')
with open(PROJECT_ROOT / CONFIG_NAME, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

paths = cfg['paths']
analysis = cfg['analysis']
graph_cfg = cfg['graph_theory']
RESULTS_DIR = (PROJECT_ROOT / paths['results_dir']).resolve()

K = int(analysis['K'])
method = analysis['method']
thresholds = np.asarray(graph_cfg['sparsity_thresholds'], dtype=float)
use_parallel = bool(graph_cfg.get('use_parallel', True))
n_jobs = graph_cfg.get('n_jobs')

with open(RESULTS_DIR / 'analysis_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

select_networks = np.array(meta['select_networks'])

fc_npz = np.load(RESULTS_DIR / '04_weighted_fc' / 'weighted_average_FC.npz', allow_pickle=True)
weighted_avg_FC = fc_npz['weighted_avg_FC'].item()

# Load graph metric functions from notebook 03 when running standalone.
try:
    calculate_graph_metrics
except NameError:
    get_ipython().run_line_magic('run', '03_graph_metrics_functions.ipynb')


consensus_runs=20, consensus_threshold=0.5


In [2]:
def process_single_subject_weighted_avg(subj_data, thresholds, sorted_networks, R):
    subj, FC = subj_data
    subj_results = []
    if FC is None or FC.shape[0] != R or np.all(np.isnan(FC)):
        return subj_results

    for threshold in thresholds:
        global_metrics_result = calculate_graph_metrics(FC, threshold, 'global', return_node_level=True)
        global_metrics = {k: v for k, v in global_metrics_result.items() if k != 'node_level'}
        node_level_metrics = global_metrics_result.get('node_level', {})
        local_metrics = {}
        lesion_global_metrics = {}

        if sorted_networks is not None:
            for net in np.unique(sorted_networks):
                net_indices = np.where(sorted_networks == net)[0]
                if len(net_indices) < 3:
                    continue
                net_FC = FC[np.ix_(net_indices, net_indices)]
                net_metrics = calculate_graph_metrics(net_FC, threshold, f'net_{net}', return_node_level=False)
                if node_level_metrics:
                    pcs = node_level_metrics['participation_coefficient'][net_indices]
                    valid_pc = pcs[np.isfinite(pcs)]
                    net_metrics[f'net_{net}_participation_coefficient'] = (
                        float(np.mean(valid_pc)) if len(valid_pc) else np.nan
                    )
                local_metrics.update(net_metrics)

                FC_lesioned = FC.copy()
                FC_lesioned[np.ix_(net_indices, np.arange(R))] = 0
                FC_lesioned[np.ix_(np.arange(R), net_indices)] = 0
                lesion_metrics = calculate_graph_metrics(
                    FC_lesioned, threshold, metric_type=f'lesion_net_{net}_global', return_node_level=False
                )
                lesion_global_metrics.update(lesion_metrics)

        row = {'subject': subj, 'threshold': float(threshold)}
        row.update(global_metrics)
        row.update(local_metrics)
        row.update(lesion_global_metrics)
        subj_results.append(row)
    return subj_results


def run_intact_graph_analysis(weighted_avg_FC, thresholds, sorted_networks, use_parallel=True, n_jobs=None, save_dir='results/intact_graph'):
    valid_subjects = [s for s, fc in weighted_avg_FC.items() if fc is not None]
    if not valid_subjects:
        raise ValueError('No valid weighted-average FC matrices.')

    R = weighted_avg_FC[valid_subjects[0]].shape[0]
    subj_data_list = [(s, weighted_avg_FC[s]) for s in valid_subjects]
    process_func = partial(process_single_subject_weighted_avg, thresholds=thresholds, sorted_networks=sorted_networks, R=R)

    if n_jobs is None:
        n_jobs = min(mp.cpu_count(), len(valid_subjects))

    all_results = []
    if use_parallel:
        with mp.Pool(processes=n_jobs) as pool:
            for subj_result in tqdm(pool.imap(process_func, subj_data_list), total=len(subj_data_list), desc='Subjects'):
                all_results.extend(subj_result)
    else:
        for subj_data in tqdm(subj_data_list, desc='Subjects'):
            all_results.extend(process_func(subj_data))

    os.makedirs(save_dir, exist_ok=True)
    results_df = pd.DataFrame(all_results)
    out_name = f'intact_graph_metrics_1000parcels_{K}states_{method}.csv'
    results_df.to_csv(Path(save_dir) / out_name, index=False)

    unified_df = calculate_unified_metrics(results_df.assign(state='weighted_average'))
    unified_name = f'intact_graph_unified_metrics_1000parcels_{K}states_{method}.csv'
    unified_df.to_csv(Path(save_dir) / unified_name, index=False)
    print(f'Saved {out_name} ({len(results_df)} rows) and {unified_name}.')
    return results_df, unified_df


save_dir = RESULTS_DIR / 'intact_graph'
results_df, unified_df = run_intact_graph_analysis(
    weighted_avg_FC,
    thresholds=thresholds,
    sorted_networks=select_networks,
    use_parallel=use_parallel,
    n_jobs=n_jobs,
    save_dir=str(save_dir),
)


Subjects:   0%|          | 0/3 [00:00<?, ?it/s]

Subjects:  33%|███▎      | 1/3 [00:00<00:00,  3.13it/s]

Subjects:  67%|██████▋   | 2/3 [00:00<00:00,  3.21it/s]

Subjects: 100%|██████████| 3/3 [00:00<00:00,  3.22it/s]

Subjects: 100%|██████████| 3/3 [00:00<00:00,  3.21it/s]

计算跨阈限的统一指标...
统一指标计算完成，共 3 行数据
Saved intact_graph_metrics_1000parcels_3states_example.csv (9 rows) and intact_graph_unified_metrics_1000parcels_3states_example.csv.
